# 04 — Spatial Metrics and Network Context

Attaches safety, traffic, school, subway, Vision Zero, and protected-network proximity metrics to the candidate segments.

Distances are calculated in **EPSG:2263 (US survey feet)**.

Traffic is retained as a supplementary metric only; missing traffic observations are **not** converted to zero. Schools and subway entrances use 500-ft proximity. Vision Zero and protected-network proximity use 100-ft buffers.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

project = Path.cwd().resolve()
if project.name == "notebooks":
    project = project.parent

raw = project / "data" / "raw"
interim = project / "data" / "interim"
processed = project / "data" / "processed"
tables_dir = project / "outputs" / "tables"

interim.mkdir(parents=True, exist_ok=True)
processed.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project)

import geopandas as gpd

In [ ]:
candidates = gpd.read_file(
    processed / "bike_candidates_nonprotected_2263.gpkg"
)
candidates["SegmentID"] = pd.to_numeric(
    candidates["SegmentID"], errors="coerce"
).astype("Int64")

crash_metrics = pd.read_csv(
    processed / "crash_segment_metrics_2022_2026.csv"
)
crash_metrics["SegmentID"] = pd.to_numeric(
    crash_metrics["SegmentID"], errors="coerce"
).astype("Int64")

candidates = candidates.merge(
    crash_metrics,
    on="SegmentID",
    how="left"
)

crash_cols = [
    "crash_count", "persons_injured", "persons_killed",
    "ped_injured", "ped_killed", "cyclist_injured",
    "cyclist_killed", "motorist_injured", "motorist_killed",
    "morning", "afternoon", "evening", "vru_injured", "vru_killed"
]
candidates[crash_cols] = candidates[crash_cols].fillna(0)

print("Candidate rows:", len(candidates))
print("Segments with crashes:", (candidates["crash_count"] > 0).sum())

## Supplementary traffic exposure

In [ ]:
traffic = pd.read_csv(processed / "traffic_daily_2022_2026.csv")
traffic["SegmentID"] = pd.to_numeric(
    traffic["SegmentID"], errors="coerce"
).astype("Int64")

traffic_segment_metrics = (
    traffic
    .groupby("SegmentID")
    .agg(
        traffic_days=("Date", "nunique"),
        morning_traffic=("morning", "median"),
        afternoon_traffic=("afternoon", "median"),
        evening_traffic=("evening", "median"),
        total_7_24_traffic=("total_7_24", "median")
    )
    .reset_index()
)

candidates = candidates.merge(
    traffic_segment_metrics,
    on="SegmentID",
    how="left"
)

print("Candidates with traffic observations:",
      candidates["total_7_24_traffic"].notna().sum())
print("Candidates without traffic observations:",
      candidates["total_7_24_traffic"].isna().sum())

## School and subway access

In [ ]:
schools = gpd.read_file(processed / "schools_2263.gpkg")

candidate_buffers_500 = candidates[["SegmentID", "geometry"]].copy()
candidate_buffers_500["geometry"] = candidate_buffers_500.geometry.buffer(500)

school_join = gpd.sjoin(
    schools,
    candidate_buffers_500,
    how="inner",
    predicate="within"
)

school_counts = (
    school_join.groupby("SegmentID")
    .size()
    .rename("school_count_500ft")
    .reset_index()
)

candidates = candidates.merge(school_counts, on="SegmentID", how="left")
candidates["school_count_500ft"] = (
    candidates["school_count_500ft"].fillna(0).astype(int)
)

print("Segments with a school within 500 ft:",
      (candidates["school_count_500ft"] > 0).sum())

In [ ]:
subway = gpd.read_file(processed / "subway_enterances_2263.gpkg")

subway_join = gpd.sjoin(
    subway,
    candidate_buffers_500,
    how="inner",
    predicate="within"
)

subway_counts = (
    subway_join.groupby("SegmentID")
    .size()
    .rename("subway_count_500ft")
    .reset_index()
)

candidates = candidates.merge(subway_counts, on="SegmentID", how="left")
candidates["subway_count_500ft"] = (
    candidates["subway_count_500ft"].fillna(0).astype(int)
)

print("Segments with a subway entrance within 500 ft:",
      (candidates["subway_count_500ft"] > 0).sum())

## Vision Zero context

In [ ]:
vz = gpd.read_file(processed / "vzv_priority_corridors_2263.gpkg")
vz_buffer = vz[["geometry"]].copy()
vz_buffer["geometry"] = vz_buffer.geometry.buffer(100)

candidate_vz_join = gpd.sjoin(
    candidates[["SegmentID", "geometry"]],
    vz_buffer,
    how="left",
    predicate="intersects"
)

vz_ids = set(
    candidate_vz_join.loc[
        candidate_vz_join["index_right"].notna(),
        "SegmentID"
    ]
)

candidates["vz_priority"] = candidates["SegmentID"].isin(vz_ids).astype(int)

print("Segments within 100 ft of a Vision Zero corridor:",
      candidates["vz_priority"].sum())

## Length filter

In [ ]:
candidates["length_ft"] = candidates.geometry.length
candidates["length_mi"] = candidates["length_ft"] / 5280

candidates = candidates[candidates["length_ft"] >= 150].copy()

print("Candidates after 150-ft minimum length:", len(candidates))
print(candidates[["length_ft", "length_mi"]].describe())

## Protected-network endpoint proximity

`connectivity_score` is an **endpoint-proximity** measure, not a fully topological network-connectivity measure:

- `0` — neither endpoint is within 100 ft of the protected network
- `1` — one endpoint is within 100 ft
- `2` — both endpoints are within 100 ft

This captures potential extensions and gap-filling links while avoiding claims of validated topological connectivity.

In [ ]:
bike = gpd.read_file(processed / "bike_routes_2263_nyc.gpkg")

protected_types = ["Protected", "Curbside", "Curbside Buffered"]
bike_on = bike[bike["onoffst"] == "ON"].copy()

protected_bike = bike_on[
    bike_on["ft_facilit"].isin(protected_types) |
    bike_on["tf_facilit"].isin(protected_types)
].copy()

protected_buffer = protected_bike[["geometry"]].copy()
protected_buffer["geometry"] = protected_buffer.geometry.buffer(100)

candidate_endpoints = candidates[["SegmentID", "geometry"]].copy()
candidate_endpoints["start_point"] = candidate_endpoints.geometry.apply(
    lambda g: g.boundary.geoms[0]
)
candidate_endpoints["end_point"] = candidate_endpoints.geometry.apply(
    lambda g: g.boundary.geoms[-1]
)

start_points = gpd.GeoDataFrame(
    candidate_endpoints[["SegmentID"]].copy(),
    geometry=candidate_endpoints["start_point"],
    crs=candidates.crs
)
end_points = gpd.GeoDataFrame(
    candidate_endpoints[["SegmentID"]].copy(),
    geometry=candidate_endpoints["end_point"],
    crs=candidates.crs
)

start_join = gpd.sjoin(
    start_points, protected_buffer, how="left", predicate="intersects"
)
end_join = gpd.sjoin(
    end_points, protected_buffer, how="left", predicate="intersects"
)

start_ids = set(
    start_join.loc[start_join["index_right"].notna(), "SegmentID"]
)
end_ids = set(
    end_join.loc[end_join["index_right"].notna(), "SegmentID"]
)

candidates["start_connected"] = candidates["SegmentID"].isin(start_ids).astype(int)
candidates["end_connected"] = candidates["SegmentID"].isin(end_ids).astype(int)
candidates["connectivity_score"] = (
    candidates["start_connected"] + candidates["end_connected"]
)
candidates["connectivity_norm"] = candidates["connectivity_score"] / 2

print(candidates["connectivity_score"].value_counts().sort_index())

In [ ]:
metrics_out = processed / "bike_candidates_with_metrics_2263.gpkg"
candidates.to_file(metrics_out, driver="GPKG")

print("Saved:", metrics_out)
print("Rows:", len(candidates))